In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===============================================================
# 🔧 0) 기본 셋업: 라이브러리 / 디바이스 / 시드 / 경로
# ===============================================================
import os, re, json, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

# ▶️ GPU ↔ CPU 전환 (True면 CUDA 사용)
USE_CUDA = True

def pick_device():
    if USE_CUDA and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            _ = torch.tensor([0.0], device='cuda')  # 스모크 테스트
            print("✅ Device: CUDA (GPU)")
            return torch.device("cuda")
        except Exception as e:
            print("⚠️ CUDA 스모크 테스트 실패, CPU로 전환:", e)
    print("✅ Device: CPU")
    return torch.device("cpu")

device = pick_device()

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

# 경로/하이퍼파라미터
FILE1 = "/content/drive/MyDrive/중간평가_이후_모델/all_preprocessed.csv"
FILE2 = "/content/drive/MyDrive/중간평가_이후_모델/merged_dialogues.csv"
LEXICON_PATH = "/content/drive/MyDrive/중간평가_이후_모델/lexicon_artifacts/LEXICON_topK_logodds.json"

MODEL_NAME = "skt/kobert-base-v1"
OUTPUT_DIR = "/content/drive/MyDrive/중간평가_이후_모델/models/kobert_lex_choi"

# 🔽 속도 최적값(대략 빠르게)
MAX_LEN      = 128
EPOCHS       = 5
BATCH_SIZE   = 32
EVAL_BS      = 128
LR           = 5e-5
W_DECAY      = 0.01
WARMUP_RATIO = 0.1
DROPOUT      = 0.1
SEED         = 42

# ✅ C-5: DataLoader 최적화
NUM_WORKERS = 2 if device.type == "cuda" else 0
PIN_MEMORY  = (device.type == "cuda")

# ✅ C-2: AMP
USE_AMP = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# 🛑 EarlyStopping
ES_PATIENCE = 1

# ⚡ 빠른 학습(시간 단축): 개수 상한
FAST_BY_COUNT = True
TRAIN_CAP     = 80_000   # ← 시간 줄이려면 80k 권장(필요시 60k/120k로 조정)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===============================================================
# 1) 데이터 분리: FILE1(+FILE2) → train/valid/test (stratify)
# ===============================================================
def _normalize_columns(df: pd.DataFrame):
    cols = [c.lower() for c in df.columns]
    df.columns = cols
    if 'text' not in df.columns and '내용' in df.columns:
        df = df.rename(columns={'내용':'text'})
    if 'label' not in df.columns and '라벨' in df.columns:
        df = df.rename(columns={'라벨':'label'})
    if 'label' not in df.columns:
        df['label'] = 0
    return df[['text','label']]

def prepare_splits(file1, file2=None, outdir=OUTPUT_DIR, test_size=0.15, valid_size=0.15, seed=SEED):
    df1 = _normalize_columns(pd.read_csv(file1))
    df1['label'] = df1['label'].astype(int)

    if file2:
        df2 = _normalize_columns(pd.read_csv(file2))
        df2['label'] = 0
        df = pd.concat([df1, df2], ignore_index=True)
    else:
        df = df1

    df['text'] = df['text'].astype(str).str.strip()
    df = df.dropna(subset=['text','label']).drop_duplicates(subset=['text'])

    train_valid, test = train_test_split(df, test_size=test_size, random_state=seed, stratify=df['label'])
    valid_ratio = valid_size / (1.0 - test_size)
    train, valid = train_test_split(train_valid, test_size=valid_ratio, random_state=seed, stratify=train_valid['label'])

    os.makedirs(outdir, exist_ok=True)
    train.to_csv(f"{outdir}/train.csv", index=False)
    valid.to_csv(f"{outdir}/valid.csv", index=False)
    test.to_csv(f"{outdir}/test.csv", index=False)
    print(f"📂 Split 완료 → train={len(train)}, valid={len(valid)}, test={len(test)}")

prepare_splits(FILE1, FILE2)

# ===============================================================
# 2) 사전(lexicon) 로드 (JSON/CSV 지원)
# ===============================================================
def load_lexicon(path):
    if path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, dict):
            return {str(k).strip(): float(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            lex = {}
            for it in obj:
                w = str(it.get("word","")).strip()
                s = float(it.get("score", 0.0))
                if w: lex[w] = s
            return lex
        else:
            raise ValueError("지원하지 않는 JSON 구조")
    else:
        df = pd.read_csv(path)
        return {str(r['word']).strip(): float(r['score']) for _, r in df.iterrows()}

lexicon = load_lexicon(LEXICON_PATH)
print("🔠 Lexicon size:", len(lexicon))

# ===============================================================
# 3) 토크나이저 준비 (PAD 보장)
# ===============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
added_pad = False
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    added_pad = True
    print("⚠️ tokenizer.pad_token 이 없어 [PAD]를 추가했습니다.")

# ===============================================================
# 4) 사전 기반 특성 (10d)
# ===============================================================
TOKEN_PAT = re.compile(r"[가-힣A-Za-z0-9]+")

def simple_tokenize(text: str):
    return TOKEN_PAT.findall(str(text))

def lexicon_features(tokens, lex):
    scores, pos_hits, neg_hits = [], 0, 0
    for t in tokens:
        s = lex.get(t)
        if s is not None:
            scores.append(s)
            if s >= 0: pos_hits += 1
            else: neg_hits += 1
    hit = len(scores)
    if hit == 0:
        return np.zeros(10, dtype=np.float32)
    arr = np.array(scores, dtype=np.float32)
    return np.array([
        hit, arr.sum(), arr.mean(), arr.max(), arr.min(), arr.std(),
        pos_hits, neg_hits, pos_hits/hit, hit/max(1,len(tokens))
    ], dtype=np.float32)

# ===============================================================
# 5) Dataset (안전판): token_type_ids=0 고정, dtype/길이 보강
# ===============================================================
class TxtDataset(Dataset):
    def __init__(self, df, tokenizer, lex, max_len):
        self.texts = df["text"].astype(str).fillna("").tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.lex = lex
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        text = str(self.texts[i]).strip()
        label = int(self.labels[i])

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
            return_token_type_ids=True,
            return_attention_mask=True
        )
        input_ids = enc.get("input_ids", torch.zeros((1, self.max_len), dtype=torch.long)).squeeze(0).to(torch.long)
        attn_mask = enc.get("attention_mask", torch.ones((1, self.max_len), dtype=torch.long)).squeeze(0).to(torch.long)
        token_type = torch.zeros((self.max_len,), dtype=torch.long)  # 한 문장: 세그먼트 전부 0

        toks = simple_tokenize(text)
        lex_feat = lexicon_features(toks, self.lex)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "token_type_ids": token_type,
            "lex_feat": torch.tensor(lex_feat, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long)
        }

# ===============================================================
# 6) 모델: KoBERT [CLS] + lexicon(10d) → 256 → 2
# ===============================================================
class FusionClassifier(nn.Module):
    def __init__(self, model_name, lex_dim=10, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.fc1 = nn.Linear(hidden + lex_dim, 256)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc_out = nn.Linear(256, 2)

    def forward(self, input_ids, attention_mask, token_type_ids, lex_feat):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:,0,:]
        x = torch.cat([cls, lex_feat], dim=-1)
        x = self.drop(self.act(self.fc1(x)))
        return self.fc_out(x)

# CPU에서 생성 → (필요 시) GPU로 이동
model = FusionClassifier(MODEL_NAME, lex_dim=10, dropout=DROPOUT)
if added_pad:
    model.backbone.resize_token_embeddings(len(tokenizer))
if device.type == "cuda":
    model = model.cuda()

# ===============================================================
# 7) 클래스 가중치 계산 (A)
# ===============================================================
def get_class_weights():
    cnt = pd.read_csv(f"{OUTPUT_DIR}/train.csv")["label"].value_counts()
    n0 = int(cnt.get(0, 1)); n1 = int(cnt.get(1, 1))
    w0 = 1.0
    w1 = max(1.0, n0 / max(1, n1))   # 대략 400~500배
    return torch.tensor([w0, w1], dtype=torch.float32, device=device)

CLASS_WEIGHTS = get_class_weights()

# ===============================================================
# 8) 빠른 학습(시간 단축) 샘플링
# ===============================================================
def make_fast_train(df: pd.DataFrame):
    if not FAST_BY_COUNT:
        return df
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    minor = df[df.label == 1]
    major = df[df.label == 0]

    keep_minor = minor  # 소수(1) 전부 확보
    remain = max(0, TRAIN_CAP - len(keep_minor))
    keep_major = major.iloc[:min(remain, len(major))]

    fast = pd.concat([keep_minor, keep_major]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(f"⚡ FAST_BY_COUNT → {len(fast):,} samples (1={len(keep_minor):,}, 0={len(keep_major):,})")
    print(fast['label'].value_counts())
    return fast

✅ Device: CUDA (GPU)


/tmp/ipython-input-1707021414.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


📂 Split 완료 → train=495641, valid=106209, test=106209
🔠 Lexicon size: 30


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

In [3]:
# ===============================================================
# 9) 학습 루프: AMP + 임베딩/세그먼트 범위 가드
# ===============================================================
def run_epoch(model, loader, optimizer, scheduler, device, train=True):
    model.train() if train else model.eval()
    loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS).to(device)

    vocab_size = model.backbone.get_input_embeddings().num_embeddings
    try:
        unk_id = model.backbone.config.unk_token_id
        if unk_id is None:
            unk_id = tokenizer.unk_token_id
        if unk_id is None:
            unk_id = 0
    except:
        unk_id = 0
    type_vocab_size = getattr(model.backbone.config, "type_vocab_size", 2)

    total_loss, preds, gts = 0.0, [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        tt   = batch["token_type_ids"].to(device, non_blocking=True)
        lex  = batch["lex_feat"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True).long()

        # 범위 가드
        bad_ids = (ids < 0) | (ids >= vocab_size)
        if bad_ids.any(): ids = ids.masked_fill(bad_ids, unk_id)
        bad_tt = (tt < 0) | (tt >= type_vocab_size)
        if bad_tt.any(): tt = tt.masked_fill(bad_tt, 0)

        optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            if USE_AMP:
                with torch.cuda.amp.autocast():
                    logits = model(ids, mask, tt, lex)
                    loss = loss_fn(logits, labels)
                if train:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    if scheduler: scheduler.step()
            else:
                logits = model(ids, mask, tt, lex)
                loss = loss_fn(logits, labels)
                if train:
                    loss.backward()
                    optimizer.step()
                    if scheduler: scheduler.step()

        total_loss += loss.item() * ids.size(0)
        preds += logits.argmax(-1).detach().cpu().tolist()
        gts   += labels.detach().cpu().tolist()

    acc = accuracy_score(gts, preds)
    p, r, f1, _ = precision_recall_fscore_support(gts, preds, average="binary", zero_division=0)
    return total_loss/len(loader.dataset), acc, p, r, f1

def evaluate_full(model, loader, device):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for b in loader:
            out = model(b["input_ids"].to(device), b["attention_mask"].to(device), b["token_type_ids"].to(device), b["lex_feat"].to(device))
            preds += out.argmax(-1).cpu().tolist()
            gts   += b["label"].tolist()
    return classification_report(gts, preds, digits=4), confusion_matrix(gts, preds)

# ===============================================================
# 10) 메인: 빠른 학습 + AMP + EarlyStopping
# ===============================================================
def main():
    # 원본(불균형) 사용
    train_df = pd.read_csv(f"{OUTPUT_DIR}/train.csv")
    valid_df = pd.read_csv(f"{OUTPUT_DIR}/valid.csv")
    test_df  = pd.read_csv(f"{OUTPUT_DIR}/test.csv")

    # ⚡ 빠른 학습 샘플링
    train_df = make_fast_train(train_df)
    print("train_df size:", len(train_df))

    train_ds = TxtDataset(train_df, tokenizer, lexicon, MAX_LEN)
    valid_ds = TxtDataset(valid_df, tokenizer, lexicon, MAX_LEN)
    test_ds  = TxtDataset(test_df,  tokenizer, lexicon, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    valid_loader = DataLoader(valid_ds, batch_size=EVAL_BS, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=EVAL_BS, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=W_DECAY)
    t_total = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(t_total*WARMUP_RATIO), num_training_steps=t_total
    )

    best_f1, best_path, bad = -1.0, f"{OUTPUT_DIR}/best.pt", 0
    for epoch in range(1, EPOCHS+1):
        tr = run_epoch(model, train_loader, optimizer, scheduler, device, train=True)
        va = run_epoch(model, valid_loader, optimizer, None, device, train=False)
        print(f"[Epoch {epoch}] Train F1={tr[4]:.4f} | Valid F1={va[4]:.4f}")

        if va[4] > best_f1:
            best_f1, bad = va[4], 0
            torch.save(model.state_dict(), best_path)
            print(f"  ✅ Best updated → F1={best_f1:.4f}")
        else:
            bad += 1
            print(f"  (no improvement: {bad}/{ES_PATIENCE})")
            if bad > ES_PATIENCE:
                print("⏹ Early stop")
                break

    # 테스트
    model.load_state_dict(torch.load(best_path, map_location=device))
    loss, acc, p, r, f1 = run_epoch(model, test_loader, optimizer, None, device, train=False)
    print(f"\n[Test] Acc={acc:.4f} P={p:.4f} R={r:.4f} F1={f1:.4f}")

    rep, cm = evaluate_full(model, test_loader, device)
    print("\n[Classification Report]\n", rep)
    print("[Confusion Matrix]\n", cm)

    tokenizer.save_pretrained(OUTPUT_DIR)

if __name__ == "__main__":
    main()


⚡ FAST_BY_COUNT → 80,000 samples (1=1,060, 0=78,940)
label
0    78940
1     1060
Name: count, dtype: int64
train_df size: 80000


/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train F1=0.0506 | Valid F1=0.1317
  ✅ Best updated → F1=0.1317


/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 2] Train F1=0.2061 | Valid F1=0.1984
  ✅ Best updated → F1=0.1984


/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 3] Train F1=0.2155 | Valid F1=0.1984
  (no improvement: 1/1)


/tmp/ipython-input-1288486370.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


KeyboardInterrupt: 